In [ ]:
import sys
from pathlib import Path
_ROOT = Path.cwd().parent.resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from    pathlib         import  Path
from    tqdm.notebook   import  tqdm, trange
from    time            import  perf_counter

import  torch
from    torch.utils.data    import  TensorDataset, DataLoader
import  matplotlib.pyplot   as  plt

from    core.utils    import  velocity_grid, relative_error
from    core.numerical.math   import  area_of_unit_sphere
from    core.numerical.solver     import  LAMBDA, one_step_RK4_classic
from    core.numerical.distribution   import  maxwellian_homogeneous, compute_moments_homogeneous
from    core.neural.collision_op  import  PRFNO


from    loader  import  load_dataset


DEVICE = torch.device('cuda:0')
base_path = _ROOT
assert base_path.exists()

In [ ]:
DIMENSION   = 3
N_LAYERS    = 4
MAX_FREQ    = 8
SEED        = 0
MODEL_TYPE  = "prfno"

DELTA_T     = 0.1
MAX_T       = 5.0
RES_T       = int(1.05+(MAX_T/DELTA_T))
RESOLUTION  = 2**5
MAX_V       = 3.0/LAMBDA

COL_COEFF   = 1/(0.5*torch.pi)
EXP_SPEED   = torch.arange(21)*0.05

BATCH_SIZE  = 1

path = base_path/f"step1__train_operators"/f"{MODEL_TYPE.upper()}{DIMENSION}D"/f"checkpoint__{MODEL_TYPE.upper()}{DIMENSION}D__n_layers{N_LAYERS:02d}__max_freq{MAX_FREQ:02d}__seed{SEED}"
model_and_history = torch.load(path/"best__state_dict__epoch0700.pth", weights_only=False, map_location='cpu')
# path = base_path/f"step1__train_operators"/f"{MODEL_TYPE.upper()}{DIMENSION}D"/f"{MODEL_TYPE.upper()}{DIMENSION}D__n_layers{N_LAYERS:02d}__max_freq{MAX_FREQ:02d}__seed{SEED}"
# model_and_history = torch.load(path/"model_and_history.pth", weights_only=False, map_location='cpu')
print(model_and_history.keys())

In [ ]:
v_grid = velocity_grid(DIMENSION, RESOLUTION, MAX_V)
f_init = maxwellian_homogeneous(v_grid, torch.ones(1, 1), torch.zeros(1, DIMENSION), torch.ones(1, 1))
f_init = f_init.reshape(1, *v_grid.shape[:-1], 1)
param = torch.zeros(1, 1)

# Generate dataloader
test_dataloader = DataLoader(TensorDataset(f_init, param), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
prfno = PRFNO(
    dim_domain      = DIMENSION,
    max_freq        = MAX_FREQ,
    dim_parameters  = 1,
    in_channels     = 1,
    hidden_channels = 32,
    out_channels    = 1,
    lift_layer      = [128, 128],
    project_layer   = [128, 128],
).to(DEVICE)
prfno.load_state_dict(model_and_history)
# prfno = PRFNO(**model_and_history['args']).to(DEVICE)
# prfno.load_state_dict(model_and_history['state_dict'])
prfno.eval()
v_grid = velocity_grid(DIMENSION, RESOLUTION, MAX_V, device=DEVICE)

In [ ]:
torch.cuda.reset_peak_memory_stats(DEVICE)
torch.cuda.synchronize(DEVICE)

elapse_time = perf_counter()
pred_list = []
with torch.inference_mode():
    for f_init, params in test_dataloader:
        f_pred   = f_init.to(DEVICE)
        params   = params.to(DEVICE)
        pred_list_batch = [f_pred.clone().cpu()]
        fwd = lambda t, f: COL_COEFF*prfno.forward(f, v_grid, params)
        for _ in range(RES_T-1):
            f_pred = one_step_RK4_classic(0.0, f_pred, DELTA_T, fwd)
            pred_list_batch.append(f_pred.clone().cpu())
        pred_list_batch = torch.stack(pred_list_batch, dim=1)
        pred_list.append(pred_list_batch)
pred_list = torch.cat(pred_list, dim=0)
elapse_time = perf_counter() - elapse_time

torch.cuda.synchronize(DEVICE)
peak_memory_fno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)

print(f"Elapsed time: {elapse_time:.4e} seconds")
print(f"Peak memory usage: {peak_memory_fno:.2f} MB")

In [ ]:
%%timeit -r 5 -n 5
pred_list = []
with torch.inference_mode():
    for f_init, params in test_dataloader:
        f_pred   = f_init.to(DEVICE)
        params   = params.to(DEVICE)
        pred_list_batch = [f_pred.clone().cpu()]
        fwd = lambda t, f: COL_COEFF*prfno.forward(f, v_grid, params)
        for _ in range(RES_T-1):
            f_pred = one_step_RK4_classic(0.0, f_pred, DELTA_T, fwd)
            pred_list_batch.append(f_pred.clone().cpu())
        pred_list_batch = torch.stack(pred_list_batch, dim=1)
        pred_list.append(pred_list_batch)
pred_list = torch.cat(pred_list, dim=0)

End of file